# CARE Rigorous Evaluation
**Systematic comparison of Control vs CARE plasticity mechanism**

This notebook runs controlled experiments with:
- Fixed random seeds for reproducibility
- Identical hyperparameters for fair comparison
- Comprehensive dead neuron tracking
- Results saved to Google Drive

## 1. Setup & Dependencies

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create experiment directory
import os
DRIVE_ROOT = '/content/drive/MyDrive/CARE_Experiments'
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/plots', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print(f'Experiment directory: {DRIVE_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Experiment directory: /content/drive/MyDrive/CARE_Experiments


In [5]:
!cd '/content/drive/MyDrive/CARE_Experiments'

In [ ]:
# ⚠️ STEP 1: Run this cell FIRST
# After running, go to Runtime → Restart runtime, then continue from next cell

# Uninstall conflicting scipy first
!pip uninstall -y scipy

# Install compatible versions
!pip install numpy==1.26.4 scipy==1.11.4 --quiet
!pip install torch torchvision --quiet
!pip install pytorch-lightning==2.1.0 lightning==2.1.0 --quiet
!pip install snntorch --quiet
!pip install tonic --quiet
!pip install pandas matplotlib seaborn --quiet

print('\n' + '='*60)
print('✅ Installation complete!')
print('⚠️  NOW: Runtime → Restart runtime')
print('Then run all cells starting from the NEXT cell (skip this one)')
print('='*60)

Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 55.4 MB/s eta 0:00:00:00:010:01m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
inequality 1.1.2 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires scipy>=1.13, but you have scipy 1.11.4 which is incompatible.
spopt 0.7.0 requires scipy>=1.12.0, but you have scipy 1.11.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
giddy 2.3.8 requires scipy>=1.12, but 

In [ ]:
# Install dependencies
!pip install torch torchvision pytorch-lightning snntorch tonic pandas matplotlib seaborn --quiet
!pip install hydra-core omegaconf --quiet

In [ ]:
# Clone CARE repository (or upload manually)
!git clone https://github.com/YOUR_USERNAME/CARE.git /content/CARE || echo 'Repo exists'
%cd /content/CARE
!git pull

Cloning into '/content/CARE'...
fatal: could not read Username for 'https://github.com': No such device or address
Repo exists
[Errno 2] No such file or directory: '/content/CARE'
/content
fatal: not a git repository (or any of the parent directories): .git


In [6]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
Memory: 15.8 GB


## 2. Experiment Configuration

In [7]:
import json
from datetime import datetime

# =====================================================
# EXPERIMENT CONFIGURATION - ALL LOGGED FOR REPRODUCIBILITY
# =====================================================
CONFIG = {
    # Experiment metadata
    'experiment_name': 'rigorous_care_evaluation',
    'timestamp': datetime.now().isoformat(),
    
    # Random seed for reproducibility
    'seed': 42,
    
    # Training hyperparameters
    'learning_rate': 1e-3,
    'batch_size': 64,
    'epochs': 30,
    'optimizer': 'Adam',
    'weight_decay': 0.0,
    
    # SNN-specific parameters
    'time_steps': 25,
    'beta': 0.95,  # LIF membrane decay
    'surrogate': 'FastSigmoid',
    'surrogate_slope': 25.0,
    
    # CARE plasticity parameters (for hybrid mode)
    'care_target_rate': 0.1,
    'care_learning_rate': 0.01,
    
    # Experiment grid
    'datasets': ['fashion_mnist', 'cifar10', 'nmnist'],
    'depths': [18, 34, 50],
    'modes': ['control', 'care'],
}

# Save config to Drive
config_path = f'{DRIVE_ROOT}/config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f'Config saved to: {config_path}')
print(json.dumps(CONFIG, indent=2))

Config saved to: /content/drive/MyDrive/CARE_Experiments/config.json
{
  "experiment_name": "rigorous_care_evaluation",
  "timestamp": "2026-02-05T21:02:36.701235",
  "seed": 42,
  "learning_rate": 0.001,
  "batch_size": 64,
  "epochs": 30,
  "optimizer": "Adam",
  "weight_decay": 0.0,
  "time_steps": 25,
  "beta": 0.95,
  "surrogate": "FastSigmoid",
  "surrogate_slope": 25.0,
  "care_target_rate": 0.1,
  "care_learning_rate": 0.01,
  "datasets": [
    "fashion_mnist",
    "cifar10",
    "nmnist"
  ],
  "depths": [
    18,
    34,
    50
  ],
  "modes": [
    "control",
    "care"
  ]
}


## 3. Core Imports & Utilities

In [8]:
import sys
sys.path.insert(0, '/content/CARE')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)

set_seed(CONFIG['seed'])
print('Seeds set for reproducibility')

INFO:lightning_fabric.utilities.seed:Seed set to 42


Seeds set for reproducibility


## 4. Enhanced Dead Neuron Tracking Callback

In [9]:
class EnhancedNeuronTracker(pl.Callback):
    """Comprehensive dead neuron and weight tracking."""
    
    def __init__(self, save_dir: str):
        super().__init__()
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.metrics_history = []
        self.prev_dead_mask = {}  # Track previous dead state for revival detection
        
    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch
        metrics = {'epoch': epoch}
        
        # Collect spike records from network
        spike_records = pl_module.network.get_spike_records() if hasattr(pl_module.network, 'get_spike_records') else {}
        
        total_dead = 0
        total_neurons = 0
        total_revived = 0
        total_died = 0
        
        for name, spikes in spike_records.items():
            if isinstance(spikes, list) and len(spikes) > 0:
                spikes = torch.cat(spikes, dim=0)
            elif isinstance(spikes, torch.Tensor):
                pass
            else:
                continue
            
            # Calculate firing rates per neuron
            if spikes.dim() >= 2:
                firing_rate = spikes.float().mean(dim=0)  # Average over batch/time
                while firing_rate.dim() > 1:
                    firing_rate = firing_rate.mean(dim=-1)
                
                # Dead = no spikes, Near-dead = < 1%
                dead_mask = (firing_rate == 0)
                near_dead_mask = (firing_rate < 0.01) & ~dead_mask
                
                num_neurons = firing_rate.numel()
                num_dead = dead_mask.sum().item()
                num_near_dead = near_dead_mask.sum().item()
                
                # Track revival and death events
                if name in self.prev_dead_mask:
                    prev_dead = self.prev_dead_mask[name]
                    revived = (prev_dead & ~dead_mask).sum().item()  # Was dead, now alive
                    died = (~prev_dead & dead_mask).sum().item()     # Was alive, now dead
                else:
                    revived = 0
                    died = 0
                
                self.prev_dead_mask[name] = dead_mask.clone()
                
                # Record metrics
                metrics[f'{name}_dead_ratio'] = num_dead / num_neurons if num_neurons > 0 else 0
                metrics[f'{name}_near_dead_ratio'] = num_near_dead / num_neurons if num_neurons > 0 else 0
                metrics[f'{name}_firing_rate_mean'] = firing_rate.mean().item()
                metrics[f'{name}_firing_rate_std'] = firing_rate.std().item()
                metrics[f'{name}_revived'] = revived
                metrics[f'{name}_died'] = died
                
                total_dead += num_dead
                total_neurons += num_neurons
                total_revived += revived
                total_died += died
        
        # Global metrics
        metrics['global_dead_ratio'] = total_dead / total_neurons if total_neurons > 0 else 0
        metrics['global_revived'] = total_revived
        metrics['global_died'] = total_died
        
        # Weight statistics
        for name, param in pl_module.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                short_name = name.replace('.', '_')[:30]
                metrics[f'{short_name}_mean'] = param.data.mean().item()
                metrics[f'{short_name}_std'] = param.data.std().item()
                metrics[f'{short_name}_min'] = param.data.min().item()
                metrics[f'{short_name}_max'] = param.data.max().item()
        
        # Log validation metrics
        metrics['val_accuracy'] = trainer.callback_metrics.get('val/accuracy', 0)
        if hasattr(metrics['val_accuracy'], 'item'):
            metrics['val_accuracy'] = metrics['val_accuracy'].item()
        
        self.metrics_history.append(metrics)
        
        # Save incrementally
        self.save_metrics()
    
    def save_metrics(self):
        df = pd.DataFrame(self.metrics_history)
        df.to_csv(self.save_dir / 'neuron_metrics.csv', index=False)
    
    def get_summary(self) -> Dict:
        if not self.metrics_history:
            return {}
        df = pd.DataFrame(self.metrics_history)
        return {
            'final_dead_ratio': df['global_dead_ratio'].iloc[-1],
            'min_dead_ratio': df['global_dead_ratio'].min(),
            'max_dead_ratio': df['global_dead_ratio'].max(),
            'total_revived': df['global_revived'].sum(),
            'total_died': df['global_died'].sum(),
            'best_accuracy': df['val_accuracy'].max() if 'val_accuracy' in df else 0,
        }

## 5. Data Loading

In [10]:
from torchvision import datasets, transforms

def get_dataloaders(dataset_name: str, batch_size: int, num_workers: int = 2):
    """Get train/test dataloaders for specified dataset."""
    
    data_root = '/content/data'
    
    if dataset_name == 'fashion_mnist':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        train_ds = datasets.FashionMNIST(data_root, train=True, download=True, transform=transform)
        test_ds = datasets.FashionMNIST(data_root, train=False, download=True, transform=transform)
        in_channels, num_classes = 1, 10
        
    elif dataset_name == 'cifar10':
        transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
        ])
        train_ds = datasets.CIFAR10(data_root, train=True, download=True, transform=transform_train)
        test_ds = datasets.CIFAR10(data_root, train=False, download=True, transform=transform_test)
        in_channels, num_classes = 3, 10
        
    elif dataset_name == 'nmnist':
        import tonic
        from tonic import transforms as tonic_transforms
        
        sensor_size = tonic.datasets.NMNIST.sensor_size
        transform = tonic_transforms.Compose([
            tonic_transforms.Denoise(filter_time=10000),
            tonic_transforms.ToFrame(sensor_size=sensor_size, n_time_bins=CONFIG['time_steps']),
        ])
        train_ds = tonic.datasets.NMNIST(save_to=data_root, train=True, transform=transform)
        test_ds = tonic.datasets.NMNIST(save_to=data_root, train=False, transform=transform)
        in_channels, num_classes = 2, 10
    else:
        raise ValueError(f'Unknown dataset: {dataset_name}')
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return train_loader, test_loader, in_channels, num_classes

## 6. Model Creation

In [11]:
os.chdir('/content/drive/MyDrive/CARE_Experiments')

In [12]:
# Import CARE models
from models.components.neuron import CareResNet
from systems.experiment import DeadNeuronExperiment

def create_model(depth: int, in_channels: int, num_classes: int, 
                 use_care: bool, num_steps: int) -> pl.LightningModule:
    """Create ResNet model with or without CARE plasticity."""
    
    # CareResNet uses block_type: 'sew' or 'ms'
    # 'sew' = SEW-ResNet (Spike Addition) - standard
    # 'ms' = MS-ResNet (Membrane Shortcut) - with CARE-like mechanism
    block_type = 'ms' if use_care else 'sew'
    
    network = CareResNet(
        depth=depth,
        in_channels=in_channels,
        num_classes=num_classes,
        num_steps=num_steps,
        beta=CONFIG['beta'],
        block_type=block_type,
    )
    
    model = DeadNeuronExperiment(
        network=network,
        num_classes=num_classes,
        num_steps=num_steps,
        learning_rate=CONFIG['learning_rate'],
    )
    
    return model

## 7. Single Experiment Runner

In [13]:
def run_single_experiment(
    dataset_name: str,
    depth: int,
    mode: str,  # 'control' or 'care'
) -> Dict:
    """Run a single controlled experiment."""
    
    exp_name = f'{dataset_name}_resnet{depth}_{mode}'
    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {exp_name}')
    print(f'{"="*60}')
    
    # Reset seeds
    set_seed(CONFIG['seed'])
    
    # Create save directory
    save_dir = Path(DRIVE_ROOT) / 'results' / exp_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Get data
    train_loader, test_loader, in_channels, num_classes = get_dataloaders(
        dataset_name, CONFIG['batch_size']
    )
    
    # Create model
    use_care = (mode == 'care')
    model = create_model(
        depth=depth,
        in_channels=in_channels,
        num_classes=num_classes,
        use_care=use_care,
        num_steps=CONFIG['time_steps'],
    )
    
    # Callbacks
    neuron_tracker = EnhancedNeuronTracker(save_dir)
    checkpoint_cb = ModelCheckpoint(
        dirpath=save_dir / 'checkpoints',
        filename='best-{epoch:02d}-{val/accuracy:.4f}',
        monitor='val/accuracy',
        mode='max',
        save_top_k=1,
    )
    
    # Trainer
    trainer = pl.Trainer(
        max_epochs=CONFIG['epochs'],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[neuron_tracker, checkpoint_cb],
        enable_progress_bar=True,
        logger=False,  # We handle logging manually
        deterministic=True,
    )
    
    # Train
    trainer.fit(model, train_loader, test_loader)
    
    # Get final results
    summary = neuron_tracker.get_summary()
    summary['experiment_name'] = exp_name
    summary['dataset'] = dataset_name
    summary['depth'] = depth
    summary['mode'] = mode
    
    # Save summary
    with open(save_dir / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f'\nResults: {summary}')
    print(f'Saved to: {save_dir}')
    
    return summary

## 8. Run Full Experiment Matrix

In [ ]:
# Run all experiments
all_results = []

for dataset in CONFIG['datasets']:
    for depth in CONFIG['depths']:
        for mode in CONFIG['modes']:
            try:
                result = run_single_experiment(dataset, depth, mode)
                all_results.append(result)
            except Exception as e:
                print(f'ERROR in {dataset}/{depth}/{mode}: {e}')
                all_results.append({
                    'dataset': dataset,
                    'depth': depth,
                    'mode': mode,
                    'error': str(e)
                })

# Save all results
results_df = pd.DataFrame(all_results)
results_df.to_csv(f'{DRIVE_ROOT}/all_results.csv', index=False)
print(f'\n\nALL RESULTS SAVED TO: {DRIVE_ROOT}/all_results.csv')
results_df

INFO:lightning_fabric.utilities.seed:Seed set to 42



EXPERIMENT: fashion_mnist_resnet18_control
 Using SEW-ResNet Blocks (Spike Addition)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


[SABOTAGE INIT] Applied with std=0.01 to depth-18 network


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name    | Type       | Params
---------------------------------------
0 | network | CareResNet | 11.2 M
---------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.663    Total estimated model params size (MB)


...

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## 9. Generate Comparison Plots

In [ ]:
import pandas as pd
def generate_comparison_plots(results_df: pd.DataFrame):
    """Generate all comparison visualizations."""
    plots_dir = Path(DRIVE_ROOT) / 'plots'
    plots_dir.mkdir(exist_ok=True)
    
    # Check available columns
    print("Available columns:", results_df.columns.tolist())
    
    # Find the accuracy column (might be named differently)
    acc_col = None
    for col in ['best_accuracy', 'val_accuracy', 'accuracy', 'test_accuracy']:
        if col in results_df.columns:
            acc_col = col
            break
    
    if acc_col is None:
        print("No accuracy column found. Available columns:", results_df.columns.tolist())
        return
    
    print(f"Using accuracy column: {acc_col}")
    
    # Filter out errors
    valid_df = results_df.dropna(subset=[acc_col]).copy()
    
    if valid_df.empty:
        print("No valid results to plot")
        return
    
    # Accuracy comparison bar chart
    datasets = valid_df['dataset'].unique()
    fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 5))
    if len(datasets) == 1:
        axes = [axes]
    
    for i, dataset in enumerate(datasets):
        subset = valid_df[valid_df['dataset'] == dataset]
        ax = axes[i]
        depths = sorted(subset['depth'].unique())
        x = np.arange(len(depths))
        width = 0.35
        
        control_acc = []
        care_acc = []
        for d in depths:
            c = subset[(subset['depth']==d) & (subset['mode']=='control')]
            h = subset[(subset['depth']==d) & (subset['mode']=='care')]
            control_acc.append(c[acc_col].values[0] if len(c) > 0 else 0)
            care_acc.append(h[acc_col].values[0] if len(h) > 0 else 0)
        
        ax.bar(x - width/2, control_acc, width, label='Control', color='#e74c3c', alpha=0.8)
        ax.bar(x + width/2, care_acc, width, label='CARE', color='#27ae60', alpha=0.8)
        
        ax.set_xlabel('ResNet Depth')
        ax.set_ylabel('Best Accuracy')
        ax.set_title(dataset.upper().replace('_', ' '))
        ax.set_xticks(x)
        ax.set_xticklabels([f'ResNet-{d}' for d in depths])
        ax.legend()
        ax.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(plots_dir / 'accuracy_comparison.png', dpi=150)
    plt.show()
    print(f'Plots saved to: {plots_dir}')

generate_comparison_plots(results_df)

NameError: name 'pd' is not defined

In [ ]:
# Check the errors
print(results_df['error'].values)

["CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "CareResNet.__init__() got an unexpected keyword argument 'use_care'"
 "Care

## 10. Statistical Analysis

In [ ]:
from scipy import stats

def compute_statistics(results_df: pd.DataFrame):
    """Compute statistical significance tests."""
    
    print('='*60)
    print('STATISTICAL ANALYSIS')
    print('='*60)
    
    for dataset in results_df['dataset'].unique():
        subset = results_df[results_df['dataset'] == dataset]
        control = subset[subset['mode'] == 'control']['best_accuracy'].values
        care = subset[subset['mode'] == 'care']['best_accuracy'].values
        
        if len(control) > 0 and len(care) > 0:
            # Paired t-test
            t_stat, p_value = stats.ttest_ind(care, control)
            
            print(f'\n{dataset.upper()}')
            print(f'  Control mean accuracy: {np.mean(control):.4f} ± {np.std(control):.4f}')
            print(f'  CARE mean accuracy:    {np.mean(care):.4f} ± {np.std(care):.4f}')
            print(f'  Difference:            {np.mean(care) - np.mean(control):.4f}')
            print(f'  t-statistic:           {t_stat:.4f}')
            print(f'  p-value:               {p_value:.4f}')
            print(f'  Significant (p<0.05):  {"YES" if p_value < 0.05 else "NO"}')

compute_statistics(results_df)

: 

## 11. Summary Report

In [ ]:
print('='*60)
print('EXPERIMENT COMPLETE')
print('='*60)
print(f'\nResults saved to: {DRIVE_ROOT}')
print(f'\nFiles:')
print(f'  - config.json: All hyperparameters')
print(f'  - all_results.csv: Summary of all experiments')
print(f'  - results/<exp>/neuron_metrics.csv: Per-epoch tracking')
print(f'  - plots/: Visualization images')
print(f'\n{results_df.to_string()}')